In [1]:
import numpy as np
from sympleq.core.symmetries.pauli import pauli_reduce
from sympleq.core.symmetries.clifford import clifford_phase_decomposition, qudit_cost
from scripts.experiments.symmetries.src.block_decomposition import block_decompose, block_decompose_optimal, block_indexes
from sympleq.models.Ising import ising_chain_hamiltonian, ising_2d_hamiltonian, heuristic_clifford_symmetry
from sympleq.core.circuits import Gate, Circuit, gate_to_circuit
import numpy as np

In [2]:

N = 4
J = 1
h = 0.5
H = ising_chain_hamiltonian(N, J, h, periodic=True)


In [3]:
F = heuristic_clifford_symmetry(N)
# print(F.symplectic)
S, T = block_decompose_optimal(F.symplectic, 2)

h_S, h_T = clifford_phase_decomposition(F.symplectic, F.phase_vector, S, T, int(H.lcm))
S_gate = Gate('S', F.qudit_indices, S, F.dimensions, h_S)
T_gate = Gate('T', F.qudit_indices, T, F.dimensions, h_T)

assert F == Circuit(F.dimensions, [T_gate.inv(), S_gate, T_gate]).composite_gate()

assert H.to_standard_form() == F.act(H).to_standard_form()
assert T_gate.act(S_gate.act(T_gate.inv().act(H))).to_standard_form() == H.to_standard_form()

assert S_gate.act(T_gate.inv().act(H)).to_standard_form() == T_gate.inv().act(H).to_standard_form()

print('Got T and S')
print('Qubit cost is ', qudit_cost(S_gate))

Got T and S
Qubit cost is  2


In [5]:
### Test F and F unitary

C_F = gate_to_circuit(F)
# C_S = gate_to_circuit(S)
# C_T = gate_to_circuit(T)
assert C_F.act(H).to_standard_form() == H.to_standard_form()

U_F = C_F.unitary().toarray()
H_hilbert = H.to_hilbert_space().toarray()
assert np.all(np.abs(U_F @ H_hilbert @ U_F.conj().T - H_hilbert) < 1e-8)

print('F passed')
### Test decomposition symmetry (Pauli-level)

H_prime = T_gate.inv().act(H)
H_rec = Circuit(F.dimensions, [T_gate.inv(), S_gate, T_gate]).act(H)

# For p=2 the gate_to_circuit Pauli correction drops odd phase components,
# so we verify invariance at the Pauli level using the reconstructed F.
assert H_rec.to_standard_form() == H.to_standard_form()
assert S_gate.act(H_prime).to_standard_form() == H_prime.to_standard_form()
print('F (via S,T) passed (Pauli check)')


F passed
F (via S,T) passed (Pauli check)


In [6]:
## test S unitary


C_S = gate_to_circuit(S_gate)
U_S = C_S.unitary().toarray()
H_prime_hilbert = H_prime.to_hilbert_space().toarray()
assert np.all(np.abs(U_S @ H_prime_hilbert @ U_S.conj().T - H_prime_hilbert) < 1e-8)

print('S passed')

S passed


In [7]:
C_np = Circuit(S_gate.dimensions, C_S[0:-1])
C_S = gate_to_circuit(S_gate)

print(C_np)


H [0]
H [1]
H [2]
H [3]
H [0]
SUM [1 0]
H [0]
H [2]
SUM [3 2]
H [2]
H_inv [0]
H_inv [1]
H_inv [2]
H_inv [3]
SWAP [1 0]
SWAP [3 2]



In [8]:
blocks = block_indexes(S_gate.symplectic)
print(blocks)

[[0, 1], [2, 3]]


In [ ]:
# now loop through the blocks - build local unitaries, diagonalise them. Build an eigenstate from the tensor product, and check it an eigenstate of H
# We can then do dynamic - for a given initial state apply the Clifford, what eigenstates does it overlap with? Evolve them.